# Programmatic Tool Calling

Normally a model calls one tool, waits for the answer, then calls the next. That is slow when a job needs many steps.
With **Programmatic Tool Calling (PTC)**, a supported model writes a small JavaScript program that calls several tools
in one go (loops, math, decisions), and it runs on OpenAI's computers.

`allowed_callers` says who may call a tool: left out means the model calls it directly (the default),
`["programmatic"]` means only the model's JavaScript program can call it.

Because the program runs on OpenAI's side, a fake `ScriptedModel` cannot prove it. We set it up offline, then watch a
real model write a program.


## Define one direct tool and one program-only tool

A programmatic tool must return a strict JSON _object_ shape, so we use a small Pydantic model.


In [1]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Define the tool
# @function_tool callback: a plain tool the model calls directly (the default allowed_callers)
@function_tool
def get_ninja_rank(name: str) -> str:
    """Look up a ninja's rank."""
    return "S"

In [2]:
from pydantic import (
    BaseModel,  # a Pydantic class that checks data has the right shape
)


# Tool return shape: a strict JSON object, which is what a programmatic tool must give back
class DangerSum(BaseModel):
    """The total danger level of the missions passed in."""

    total: int

In [3]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Define the tool
# @function_tool callback: the program-only tool, called from the model's JavaScript
@function_tool(allowed_callers=["programmatic"])
def sum_mission_danger_levels(levels: list[int]) -> DangerSum:
    """Sum a list of mission danger levels."""
    return DangerSum(total=sum(levels))

## Turn PTC on with `ProgrammaticToolCallingTool()`

Add it to `tools=` next to the tools a program may call. Three rules the SDK only checks when the agent **runs**, not
when you build it: use at most one `ProgrammaticToolCallingTool()`, give it at least one programmatic tool, and never
mix `needs_approval=True` with programmatic access (a human checkpoint should not be triggered by a program running
alone on the server — keep those tools direct-only). A plain `-> int` return type is rejected right away with `UserError`.


In [4]:
from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
    ProgrammaticToolCallingTool,  # lets the model write a small script that calls other tools
)

# Initialize the agent
ptc_agent = Agent(
    name="Leaf Village Assistant",
    instructions="Use your tools. Answer briefly.",
    tools=[get_ninja_rank, sum_mission_danger_levels, ProgrammaticToolCallingTool()],
)
# proves allowed_callers marks the program-only tool, and the direct tool keeps the default
print("[get_ninja_rank.allowed_callers]", get_ninja_rank.allowed_callers)
print("[sum_mission_danger_levels.allowed_callers]", sum_mission_danger_levels.allowed_callers)

[get_ninja_rank.allowed_callers] None
[sum_mission_danger_levels.allowed_callers] ['programmatic']


## Break it on purpose — `ProgrammaticToolCallingTool()` alone is not enough

`ProgrammaticToolCallingTool()` only switches the feature on. It has nothing to run without a tool the program may call,
so an agent with **only** that tool fails with `UserError`. A direct-only tool like `get_ninja_rank` does not fix it
either — the tool must list `"programmatic"` in `allowed_callers`.

The check runs when the request is built for an OpenAI Responses model, so `ScriptedModel` would never trigger it. We use
a real `OpenAIResponsesModel` with a fake key and an address nobody listens on: the error is raised _before_ any network
call, so this cell needs no real key.


In [5]:
from openai import (
    AsyncOpenAI,  # the OpenAI API client; here it never really connects
)

from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
    OpenAIResponsesModel,  # the model class that talks to the OpenAI Responses API
    ProgrammaticToolCallingTool,  # lets the model write a small script that calls other tools
    Runner,  # executes an Agent: run / run_sync / run_streamed
)
from agents.exceptions import (
    UserError,  # raised when the way you set up the SDK is wrong
)


# Demo: an agent with only ProgrammaticToolCallingTool() fails before any request is sent
async def demo_ptc_alone_fails() -> None:
    # Initialize the model
    model = OpenAIResponsesModel(
        model="gpt-5.5",
        # fake key + dead address: proves the error comes from the SDK check, not from the network
        openai_client=AsyncOpenAI(api_key="sk-fake", base_url="http://127.0.0.1:9"),
    )
    # Initialize the agent
    broken_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Use your tools. Answer briefly.",
        model=model,
        tools=[ProgrammaticToolCallingTool()],
    )
    # Run the Agent
    try:
        await Runner.run(
            broken_agent,
            "Sum the danger levels [3, 7, 2, 9].",
            max_turns=1,
        )
    except UserError as error:
        # proves PTC with no programmatic tool is rejected at run time with a clear UserError
        print("[UserError]", error)


await demo_ptc_alone_fails()
# Outside a notebook: asyncio.run(demo_ptc_alone_fails())

Error getting response


[UserError] ProgrammaticToolCallingTool() requires at least one tool whose allowed_callers includes 'programmatic', a ToolSearchTool(), or an opaque prompt-managed tool surface.


## Watch a real model write a program — needs a real key

The model writes JavaScript that calls the program-only tool. A `caller` tag on each tool output shows who made the
call: `program` or `direct`. Skipped when there is no `OPENAI_API_KEY`.


In [6]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    RunConfig,  # per-run overrides: model, workflow_name, group_id
    Runner,  # executes an Agent: run / run_sync / run_streamed
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: a real model writes JavaScript for the programmatic tool, and we show who made each call
async def demo_live_program() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Run the Agent
    result = await Runner.run(
        ptc_agent,
        "Sum the danger levels [3, 7, 2, 9] using your tools, then tell me Naruto Uzumaki's rank.",
        run_config=RunConfig(workflow_name="e15-live"),
    )
    # Inspect the result
    for item in result.new_items:
        raw = item.raw_item
        if isinstance(raw, dict):
            if item.type == "tool_call_output_item":
                # a "caller" entry only shows up when the call came from a PTC program
                caller = raw["caller"]["type"] if "caller" in raw else "direct"
                print(f"[tool output] {raw['output']!r} called by: {caller}")
        elif raw.type == "program":
            # proves the live model wrote real JavaScript
            print("[program code]", raw.code.strip())
    print("[final_output]", result.final_output)


await demo_live_program()
# Outside a notebook: asyncio.run(demo_live_program())

[tool output] 'S' called by: direct
[program code] const r = await tools.sum_mission_danger_levels({levels:[3,7,2,9]}); text(r)
[tool output] '{"total":21}' called by: program
[final_output] Danger total: **21**. Naruto Uzumaki’s rank: **S**.


## Summary

- PTC lets the model write one small program that calls many tools at once. `allowed_callers` decides who may call a
  tool, and a programmatic tool must return a strict JSON object shape.
- Use one `ProgrammaticToolCallingTool()` plus at least one programmatic tool. The SDK checks this only when the agent
  runs, and approval-gated tools should stay direct-only.
